# 9. Measuring Retrieval Quality

**RAG Pipeline Series — Notebook 9**

Notebooks 7 and 8 compared BM25, dense, and hybrid retrieval on two example queries and eyeballed where the correct chunk landed. That's fine for building intuition, but it doesn't scale past a handful of queries and it can't tell you *how much* better one retriever is than another. `rag.pdf`'s own Chapter 7 ("Retrieval Evaluation") opens with exactly this problem: *"You cannot improve what you cannot measure."*

This notebook builds a small **labeled evaluation set** — queries paired with the chunk that actually answers them — and computes the standard retrieval metrics Chapter 7 covers: **Precision@K**, **Recall@K**, **MRR**, **MAP**, and **NDCG@K**. We then run BM25, dense, and hybrid retrieval (from notebooks 7-8) through the same harness and get an actual scoreboard instead of two anecdotes.

In this notebook we will:
1. Rebuild the three retrievers from notebooks 7-8 (BM25, dense, RRF hybrid).
2. Implement Precision@K, Recall@K, Reciprocal Rank, Average Precision, and NDCG@K from their definitions.
3. Sanity-check every function against the worked example `rag.pdf` itself provides in Chapter 7.
4. Build an 8-query evaluation set spanning multiple chapters of `rag.pdf`, each paired with its one known-relevant chunk.
5. Score all three retrievers on the same eval set and compare.

## Setup

In [ ]:
%pip install -q -U langchain langchain-classic langchain-community langchain-core rank_bm25 sentence-transformers langchain-huggingface langchain-chroma chromadb pandas

## 1. Recap: the three retrievers from notebooks 7-8

Same chapter-tagged chunks, same `BM25Retriever`, same Chroma-backed dense retriever, same RRF-based `EnsembleRetriever`.

In [ ]:
from rag_utils import maybe_colab_upload

# Only runs inside Colab. Opens a file picker; select rag.pdf.
# Safe to skip this cell if you're running locally and already have the file on disk.
maybe_colab_upload()

In [1]:
from rag_utils import build_chroma_store, get_embedder, load_chapter_chunks
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

pages, full_text, chapters, chunks = load_chapter_chunks()

keyword_retriever = BM25Retriever.from_documents(chunks)
keyword_retriever.k = 10

embeddings = get_embedder()
vectorstore = build_chroma_store(chunks, embeddings=embeddings, collection_name="rag_pdf_chapters")
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

hybrid_retriever = EnsembleRetriever(retrievers=[keyword_retriever, dense_retriever], weights=[0.5, 0.5])

retrievers = {"BM25 (keyword)": keyword_retriever, "Chroma (dense)": dense_retriever, "RRF (hybrid)": hybrid_retriever}
print(f"{len(chunks)} chunks indexed into all three retrievers")

d:\youtube\TheAIGuy\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 55/55 [00:03<00:00, 14.91it/s]


181 chunks indexed into all three retrievers


## 2. The metrics, from Chapter 7's own definitions

Quoting `rag.pdf` Chapter 7 directly:

- **Precision** = `|Relevant Retrieved| / |Retrieved|` — of what you returned, how much was relevant.
- **Recall** = `|Relevant Retrieved| / |Total Relevant|` — of what's relevant, how much did you find. Chapter 7 calls this *"the primary metric for RAG retrieval evaluation"*, since a relevant chunk missing from the top-K means the LLM never sees it, and "no amount of prompt engineering will fix" that.
- **MRR** (Mean Reciprocal Rank) — `1 / rank` of the *first* relevant result, averaged across queries. Best suited to "I only need one right answer" scenarios.
- **MAP** (Mean Average Precision) — for each query, average the running Precision@k at every rank where a relevant document appears, then mean that across queries. Rewards ranking *all* relevant documents highly, not just the first one.
- **NDCG@K** (Normalized Discounted Cumulative Gain) — discounts each hit by `log2(rank + 1)` (so a hit at rank 1 counts more than a hit at rank 10), normalized against the best possible ordering. The metric of choice when relevance is graded rather than binary — we use the binary case here, but the formula generalizes directly.

We implement all five against a general `relevant_ids` set (more than one relevant chunk per query is supported), even though the eval set built below only uses one relevant chunk per query.

In [2]:
import math


def precision_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return hits / k


def recall_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return hits / len(relevant_ids)


def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def average_precision(retrieved_ids, relevant_ids):
    hits, running_precisions = 0, []
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            hits += 1
            running_precisions.append(hits / rank)
    return sum(running_precisions) / len(relevant_ids) if relevant_ids else 0.0


def ndcg_at_k(retrieved_ids, relevant_ids, k):
    def dcg(ids):
        return sum(1.0 / math.log2(rank + 1) for rank, doc_id in enumerate(ids, start=1) if doc_id in relevant_ids)

    actual = dcg(retrieved_ids[:k])
    ideal = dcg(list(relevant_ids)[:k])  # best possible ordering: every relevant doc ranked first
    return actual / ideal if ideal > 0 else 0.0

## 3. Sanity-checking against `rag.pdf`'s own worked example

Chapter 7's "Concrete Example" section gives a fully worked numeric example: a query with 3 relevant documents in the knowledge base, where the system returns `[doc_A(relevant), doc_B(irrelevant), doc_C(relevant), doc_D(irrelevant), doc_E(relevant)]`, and states the expected results directly: `Precision@5 = 0.6`, `Recall@5 = 1.0`, `MRR = 1.0`, and `MAP = mean(1/1, 2/3, 3/5)`. We can run that exact example through the functions above as a correctness check before trusting them on real chunks.

In [3]:
retrieved_example = ["doc_A", "doc_B", "doc_C", "doc_D", "doc_E"]
relevant_example = {"doc_A", "doc_C", "doc_E"}

print("Precision@5:", precision_at_k(retrieved_example, relevant_example, 5), "(rag.pdf says 0.6)")
print("Recall@5:   ", recall_at_k(retrieved_example, relevant_example, 5), "(rag.pdf says 1.0)")
print("MRR:        ", reciprocal_rank(retrieved_example, relevant_example), "(rag.pdf says 1.0)")
print("MAP:        ", round(average_precision(retrieved_example, relevant_example), 3), "(rag.pdf says mean(1/1, 2/3, 3/5) = 0.756)")

Precision@5: 0.6 (rag.pdf says 0.6)
Recall@5:    1.0 (rag.pdf says 1.0)
MRR:         1.0 (rag.pdf says 1.0)
MAP:         0.756 (rag.pdf says mean(1/1, 2/3, 3/5) = 0.756)


All four match the textbook's own numbers — the implementations above are computing exactly what Chapter 7 defines, not just something similarly named.

## 4. Building a labeled evaluation set from `rag.pdf`

An eval set needs, for every query, the ID of the chunk that actually answers it. We reuse the pattern from notebooks 3-8 (locate a chunk by a short, unique substring of its real text) across **eight queries spanning eight different chapters** — including the two example queries used throughout this series — so the comparison isn't just testing one topic. `rag_utils.build_eval_set()` resolves these substrings against the current chunking, so notebook 10 can rebuild the exact same eval set without copy-pasting it.

In [4]:
from rag_utils import build_eval_set

eval_set = build_eval_set(chunks)

for item in eval_set:
    print(f"chapter {chunks[item['relevant_idx']].metadata['chapter_num']:>2}  {item['query']}")

chapter 02  What does the Okapi BM25 formula account for besides term frequency?
chapter 01  How can giving a language model outside documents stop it from making things up?
chapter 06  What kind of queries is keyword search the strongest at handling?
chapter 06  Why is reciprocal rank fusion considered more robust than manually tuning weights?
chapter 07  Which retrieval metric matters most for a RAG system with a fixed context window?
chapter 08  Why can't a bi-encoder capture fine-grained relevance between a query and a document?
chapter 04  Give an example of two sentences that mean the same thing but share no words.
chapter 12  When should a team pick RAG over fine-tuning a model?


## 5. Evaluation harness

For each query, run a retriever, translate its returned `Document`s back into chunk indices (matching on `page_content`, same trick used in notebooks 3-8's `rank_of`), then average every metric across all eight queries.

In [5]:
content_to_idx = {d.page_content: i for i, d in enumerate(chunks)}


def evaluate_retriever(retriever, eval_set, k=5):
    metrics = {"precision@k": [], "recall@k": [], "mrr": [], "map": [], "ndcg@k": []}
    for item in eval_set:
        retrieved_docs = retriever.invoke(item["query"])
        retrieved_ids = [content_to_idx[d.page_content] for d in retrieved_docs]
        relevant_ids = {item["relevant_idx"]}

        metrics["precision@k"].append(precision_at_k(retrieved_ids, relevant_ids, k))
        metrics["recall@k"].append(recall_at_k(retrieved_ids, relevant_ids, k))
        metrics["mrr"].append(reciprocal_rank(retrieved_ids, relevant_ids))
        metrics["map"].append(average_precision(retrieved_ids, relevant_ids))
        metrics["ndcg@k"].append(ndcg_at_k(retrieved_ids, relevant_ids, k))

    return {name: round(sum(values) / len(values), 3) for name, values in metrics.items()}

## 6. Scoring BM25, dense, and hybrid retrieval

Same eval set, same `k=5`, run through all three retrievers built in section 1.

In [6]:
import pandas as pd

results = pd.DataFrame({name: evaluate_retriever(retriever, eval_set, k=5) for name, retriever in retrievers.items()}).T
results

,precision@k,recall@k,mrr,map,ndcg@k
BM25 (keyword),0.15,0.75,0.625,0.625,0.658
Chroma (dense),0.10,0.50,0.354,0.354,0.391
RRF (hybrid),0.15,0.75,0.491,0.491,0.540


With only one relevant chunk per query, `recall@k` here is just "was the right chunk anywhere in the top-5" (0 or 1 per query, averaged), and `precision@k` is that same hit divided by `k` — both are simplified versions of the general definitions once there's a single relevant document, which is exactly why Chapter 7 recommends richer eval sets (multiple relevant documents per query) for MAP and NDCG to be genuinely informative rather than redundant with recall.

## 7. Where each retriever wins or loses

Breaking the aggregate table down per-query shows *which kinds* of queries drive each retriever's score — the same keyword-vs-paraphrase pattern from notebooks 3/4/7, now visible as a metric instead of an anecdote.

In [7]:
rows = []
for item in eval_set:
    row = {"query": item["query"][:60] + ("..." if len(item["query"]) > 60 else "")}
    for name, retriever in retrievers.items():
        retrieved_ids = [content_to_idx[d.page_content] for d in retriever.invoke(item["query"])]
        row[name] = reciprocal_rank(retrieved_ids, {item["relevant_idx"]})
    rows.append(row)

pd.DataFrame(rows).set_index("query")

,BM25 (keyword),Chroma (dense),RRF (hybrid)
query,,,
What does the Okapi BM25 formula account for besides term fr...,1.0,1.000000,1.000000
How can giving a language model outside documents stop it fr...,0.5,0.000000,0.200000
What kind of queries is keyword search the strongest at hand...,0.0,0.333333,0.142857
Why is reciprocal rank fusion considered more robust than ma...,1.0,0.500000,1.000000
Which retrieval metric matters most for a RAG system with a ...,1.0,0.000000,0.250000
Why can't a bi-encoder capture fine-grained relevance betwee...,0.5,0.000000,0.333333
Give an example of two sentences that mean the same thing bu...,0.0,0.000000,0.000000
When should a team pick RAG over fine-tuning a model?,1.0,1.000000,1.000000


## Takeaways

- Precision, Recall, MRR, MAP, and NDCG each answer a different question — "how clean is the top-K," "did I find everything," "how fast was the first hit," "how good is the whole ranking," and "how good is the whole ranking, discounted by position," respectively. Recall@K is `rag.pdf`'s own recommended primary metric for RAG specifically, since a relevant chunk absent from the top-K is unrecoverable downstream.
- Validating a metrics implementation against a known worked example (Chapter 7's `doc_A`...`doc_E` case) before trusting it on real data is cheap insurance against a subtly wrong formula.
- An eval set is only as good as its labels — every query here maps to exactly one hand-verified relevant chunk. Real production eval sets usually need several relevant documents per query, which is where MAP and NDCG stop being redundant with plain recall.
- Aggregate metrics can hide *why* a retriever wins — breaking results down per query (section 7) is what actually tells you whether to invest in better keyword matching, better embeddings, or hybrid fusion.

**Next up (notebook 10):** `rag.pdf`'s own Chapter 8 argues re-ranking is *"one of the highest-ROI optimizations in a RAG pipeline"* — we'll add a cross-encoder re-ranking stage on top of these retrievers and watch it re-order a shortlist using the metrics built here.